In [115]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [149]:
def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def robustcheckpowerU(a,R,r,c,p,m,r_f,rav):    ### actually not being used
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value<=c)

def riskcalc(a,R,p,alfa,r_f):
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk + extra - (1-sum(a))*r_f
    print("the nominal risk of a:", risk)

In [141]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def ranktoset (A):
    A = list(A)
    sets = [[[A[0]]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append([new])
    return(sets)

def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-1:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominalpowerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    f_obj = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1)- (1-cp.sum(a))*r_f + z4 + z2 <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f <= c)  




In [142]:
def cutting_plane(R,r,c,p,m,r_f,sets,rav):
    nonstop = True
    iterations = 1
    while nonstop == True:
        realsets = convertlist(sets)
        #print(solvenominalpowerU(realsets,p,R,r,m,r_f,c,rav))
        [a,obj] = solvenominalpowerU(realsets,p,R,r,m,r_f,c,rav)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makeset(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [111]:
np.random.seed(5)

In [169]:
N=5
p = (np.zeros(N)+1)*1/N
I = 5
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.19730151 0.03247688 0.04068598 0.01874075 0.04883691]
[[-0.2294148   0.22453092  0.00957636 -0.06967199  0.00131606]
 [ 0.46770294  0.11938387  0.19914539  0.20538152  0.25368423]
 [ 0.26227029 -0.09209329  0.00696244 -0.10215206 -0.09223265]
 [ 0.27830155 -0.05035111  0.03416973 -0.08856527 -0.06868055]
 [ 0.20764759 -0.039086   -0.04642404  0.14871153  0.15009747]]


In [159]:
gam = 0.5
rav = 1-gam
r = 0
m = 0.05    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 0
sets = ranktoset(np.arange(N))
[a,obj,itera] = cutting_plane(R,r,c,p,m,r_f,sets,rav)
print(cutting_plane(R,r,c,p,m,r_f,sets,rav))

(array([-8.24652933e-10, -4.83552039e-10,  2.90805811e-01,  4.74005040e-01,
        2.35189148e-01]), 0.37169975675649, 1)


In [160]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
print(solvenominalpowerU (psets,p,R,r,m,r_f,c,rav))
#print(robustcheckpowerU(a,R,r,c,p,m,r_f,rav))


(array([9.21746701e-09, 6.81366730e-09, 2.90822115e-01, 4.73990745e-01,
       2.35187178e-01]), 0.3716997541805802)


In [145]:
sets

[[[0], [4]],
 [[0, 1], [4, 2]],
 [[0, 1, 2], [4, 2, 1]],
 [[0, 1, 2, 3], [4, 2, 1, 3]],
 [[0, 1, 2, 3, 4]]]

In [36]:
def phi_div(p,q,r):
    phi_cons = 0
    for i in range(len(p)):
        phi_cons = q[i]*np.log(q[i]/p[i])+phi_cons
    print(phi_cons <= r)
    print(phi_cons)

In [170]:
riskcalc(a,R,p,m,r_f)

the nominal risk of a: -0.02258366119918223


In [171]:
R.dot(a)

array([-0.02993015,  0.21492799, -0.06808708, -0.04819557,  0.09228921])